# Assignment 1 — Gradio dashboard

You're using a notebook for this stage. If you'd rather use a script,
write `app.py` instead and delete this file — `main.py` refuses to run if
it finds both `app.py` and `app.ipynb`, so exactly one of them must exist.

Replace this cell and everything below it with your own code. When run via
`uv run python main.py app`, this notebook must build and launch a Gradio
app (a `demo` that calls `.launch()`) using only what your training stage
already produced — it must never retrain anything itself. At minimum, the
dashboard must let you:

- Compare your three trained models via a prediction-vs-actual plot.
- See feature and/or target distributions.
- For classification: move a decision-threshold slider and watch the
  confusion matrix, and a business-cost number tied to your `REPORT.md`'s
  framing, change with it.

How you structure the code beyond that — cell layout, function
boundaries, naming — is your call to make and be able to defend.

## Libraries

In [1]:
import json
import os

import gradio as gr
import joblib
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score, roc_curve
from torch import nn

## Loads Saved Files and Adds formatting 

In [2]:
MODEL_NAMES = ["baseline", "sklearn", "manual_torch", "standard_torch"]
COLORS = {"baseline": "#888888", "sklearn": "#1f77b4", "manual_torch": "#d62728", "standard_torch": "#2ca02c"}


with open("models/results.json") as f:
    saved = json.load(f)
with open("models/preprocessing.json") as f:
    feature_cols = json.load(f)["feature_cols"]

DEFAULT_FN_COST = saved["config"]["fn_cost"]
DEFAULT_FP_COST = saved["config"]["fp_cost"]
BASE_RATE = saved["config"]["base_rate"]
VAL_THRESHOLDS = {name: saved["results"][name]["threshold"] for name in MODEL_NAMES}

test = pd.read_csv("data/test.csv")
X_test = test[feature_cols].values.astype("float32")
y_test = test["Default"].values.astype(int)

if os.path.exists("data/test_raw.csv"):
    raw = pd.read_csv("data/test_raw.csv")
    RAW_NOTE = "Showing original (unscaled) units."
else:
    raw = test
    RAW_NOTE = "data/test_raw.csv not found, showing scaled values."

sk_model = joblib.load("models/sklearn_logreg.joblib")
manual = torch.load("models/manual_torch.pt")
standard = nn.Linear(X_test.shape[1], 1)
standard.load_state_dict(torch.load("models/standard_torch.pt"))
standard.eval()

Xt = torch.tensor(X_test)
with torch.no_grad():
    probs = {
        "baseline": np.full(len(y_test), BASE_RATE),
        "sklearn": sk_model.predict_proba(X_test)[:, 1],
        "manual_torch": torch.sigmoid(Xt @ manual["w"] + manual["b"]).numpy(),
        "standard_torch": torch.sigmoid(standard(Xt)).squeeze(1).numpy(),
    }



## Confusion Matrix and Comparison Table

In [3]:
def confusion(y_true, proba, threshold):
    pred = proba >= threshold
    tn = int(np.sum((y_true == 0) & ~pred))
    fp = int(np.sum((y_true == 0) & pred))
    fn = int(np.sum((y_true == 1) & ~pred))
    tp = int(np.sum((y_true == 1) & pred))
    return tn, fp, fn, tp


def total_cost(y_true, proba, threshold, fn_cost, fp_cost):
    _, fp, fn, _ = confusion(y_true, proba, threshold)
    return fn_cost * fn + fp_cost * fp


def results_table():
    rows = []
    for name in MODEL_NAMES:
        r = saved["results"][name]
        rows.append({
            "model": name,
            "ROC-AUC": round(r["roc_auc"], 4),
            "PR-AUC": round(r["pr_auc"], 4),
            "threshold (chosen on validation)": round(r["threshold"], 2),
            "total cost": round(r["cost"], 1),
            "cost per loan": round(r["cost_per_loan"], 4),
            "false negatives": r["fn"],
            "false positives": r["fp"],
        })
    return pd.DataFrame(rows)


## Plots and Graphs

In [4]:
def plot_pred_vs_actual(model):
    p = probs[model]
    fig, ax = plt.subplots(figsize=(7, 4))
    bins = np.linspace(0, 1, 41)
    ax.hist(p[y_test == 0], bins=bins, alpha=0.6, label="actual: repaid (0)", color="#1f77b4")
    ax.hist(p[y_test == 1], bins=bins, alpha=0.6, label="actual: default (1)", color="#d62728")
    ax.axvline(VAL_THRESHOLDS[model], color="k", linestyle="--", label="validation threshold")
    ax.set_xlabel("predicted probability of default")
    ax.set_ylabel("number of loans")
    ax.set_title(f"Predicted vs. actual: {model}")
    ax.legend()
    fig.tight_layout()
    return fig


def plot_roc():
    fig, ax = plt.subplots(figsize=(6, 5))
    for name in MODEL_NAMES:
        fpr, tpr, _ = roc_curve(y_test, probs[name])
        auc = roc_auc_score(y_test, probs[name])
        ax.plot(fpr, tpr, label=f"{name} (AUC {auc:.3f})", color=COLORS[name])
    ax.plot([0, 1], [0, 1], color="lightgray", linestyle=":")
    ax.set_xlabel("false positive rate")
    ax.set_ylabel("true positive rate")
    ax.set_title("ROC curves on the test set")
    ax.legend()
    fig.tight_layout()
    return fig


def plot_calibration(model):
    p = probs[model]
    edges = np.linspace(0, 1, 11)
    idx = np.clip(np.digitize(p, edges) - 1, 0, 9)
    xs, ys = [], []
    for b in range(10):
        m = idx == b
        if m.sum() >= 20:
            xs.append(p[m].mean())
            ys.append(y_test[m].mean())
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot([0, 1], [0, 1], color="lightgray", linestyle=":", label="perfectly calibrated")
    ax.plot(xs, ys, marker="o", color=COLORS[model], label=model)
    ax.set_xlabel("mean predicted probability")
    ax.set_ylabel("observed default rate")
    ax.set_title(f"Calibration: {model}")
    ax.legend()
    fig.tight_layout()
    return fig


def plot_feature(feature):
    fig, ax = plt.subplots(figsize=(7, 4))
    values = raw[feature].values
    n_unique = len(np.unique(values))
    bins = min(30, n_unique) if n_unique > 1 else 1
    ax.hist(values[y_test == 0], bins=bins, alpha=0.6, density=True, label="repaid (0)", color="#1f77b4")
    ax.hist(values[y_test == 1], bins=bins, alpha=0.6, density=True, label="default (1)", color="#d62728")
    ax.set_xlabel(feature)
    ax.set_ylabel("density")
    ax.set_title(f"Distribution of {feature} by outcome")
    ax.legend()
    fig.tight_layout()
    return fig


def plot_target():
    counts = np.bincount(y_test, minlength=2)
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.bar(["repaid (0)", "default (1)"], counts, color=["#1f77b4", "#d62728"])
    for i, c in enumerate(counts):
        ax.text(i, c, f"{c}\n({c / counts.sum():.1%})", ha="center", va="bottom")
    ax.set_ylabel("number of loans")
    ax.set_title("Target distribution (test set)")
    ax.set_ylim(0, counts.max() * 1.2)
    fig.tight_layout()
    return fig


def threshold_view(model, threshold, fn_cost, fp_cost):
    p = probs[model]
    tn, fp, fn, tp = confusion(y_test, p, threshold)
    cost = fn_cost * fn + fp_cost * fp

    cm = np.array([[tn, fp], [fn, tp]]) #heatmap
    fig_cm, ax = plt.subplots(figsize=(4.5, 4))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1], ["predict repay", "predict default"])
    ax.set_yticks([0, 1], ["actual repaid", "actual default"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=14)
    ax.set_title(f"Confusion matrix at threshold {threshold:.2f}")
    fig_cm.tight_layout()


    grid = np.linspace(0.01, 0.99, 99) #cost curve
    costs = [total_cost(y_test, p, t, fn_cost, fp_cost) for t in grid]
    best_t = grid[int(np.argmin(costs))]
    fig_cost, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(grid, costs, color=COLORS[model])
    ax2.axvline(threshold, color="k", linestyle="--", label=f"selected ({threshold:.2f})")
    ax2.axvline(VAL_THRESHOLDS[model], color="gray", linestyle=":", label=f"validation choice ({VAL_THRESHOLDS[model]:.2f})")
    ax2.scatter([threshold], [cost], color="k", zorder=3)
    ax2.set_xlabel("decision threshold")
    ax2.set_ylabel("total business cost")
    ax2.set_title("Cost vs. threshold (test set)")
    ax2.legend()
    fig_cost.tight_layout()

    n = len(y_test)
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    never_approve_cost = fp_cost * (n - int(y_test.sum()))
    approve_all_cost = fn_cost * int(y_test.sum())
    summary = (
        f"### Total business cost: **{cost:,.1f}**  ({cost / n:.4f} per loan)\n\n"
        f"- False negatives (approved loans that defaulted): **{fn}** x {fn_cost:g} = {fn * fn_cost:,.1f}\n"
        f"- False positives (declined loans that would have been repaid): **{fp}** x {fp_cost:g} = {fp * fp_cost:,.1f}\n"
        f"- Recall (defaults caught): {recall:.1%}, precision: {precision:.1%}\n"
        f"- Cheapest threshold on this test set for these costs: {best_t:.2f} (cost {min(costs):,.1f})\n"
        f"- Reference: approve everyone costs {approve_all_cost:,.1f}; decline everyone costs {never_approve_cost:,.1f}"
    )
    return fig_cm, fig_cost, summary



## "Actual" Gradio UI

In [5]:
with gr.Blocks(title="Loan default dashboard") as demo:
    gr.Markdown(
        "# Loan default: three logistic regressions compared\n"
        "Decision supported: approve or decline a loan at application time. "
        "All models are loaded from disk; nothing is retrained here."
    )

    with gr.Tab("Model comparison"):
        gr.Markdown("Test-set results. Each model's threshold was chosen on the validation set by minimizing cost.")
        gr.Dataframe(value=results_table(), interactive=False)
        with gr.Row():
            model_pick = gr.Dropdown(MODEL_NAMES, value="sklearn", label="Model")
        with gr.Row():
            pva_plot = gr.Plot(label="Prediction vs. actual")
            calib_plot = gr.Plot(label="Calibration")
        roc_plot = gr.Plot(value=plot_roc(), label="ROC curves")
        model_pick.change(plot_pred_vs_actual, model_pick, pva_plot)
        model_pick.change(plot_calibration, model_pick, calib_plot)

    with gr.Tab("Distributions"):
        gr.Markdown(RAW_NOTE)
        with gr.Row():
            feature_pick = gr.Dropdown(feature_cols, value=feature_cols[0], label="Feature")
        with gr.Row():
            feat_plot = gr.Plot(label="Feature distribution")
            target_plot = gr.Plot(value=plot_target(), label="Target distribution")
        feature_pick.change(plot_feature, feature_pick, feat_plot)

    with gr.Tab("Threshold and cost"):
        gr.Markdown(
            "Move the threshold to see how the confusion matrix and the business cost change. "
            "Change the cost inputs to test other assumptions than the default."
        )
        with gr.Row():
            t_model = gr.Dropdown(MODEL_NAMES, value="sklearn", label="Model")
            t_slider = gr.Slider(0.01, 0.99, value=VAL_THRESHOLDS["sklearn"], step=0.01, label="Decision threshold")
        with gr.Row():
            fn_in = gr.Number(value=DEFAULT_FN_COST, label="Cost of a false negative (approve a loan that defaults)")
            fp_in = gr.Number(value=DEFAULT_FP_COST, label="Cost of a false positive (decline a good loan)")
        summary_md = gr.Markdown()
        with gr.Row():
            cm_plot = gr.Plot(label="Confusion matrix")
            cost_plot = gr.Plot(label="Cost curve")
        t_inputs = [t_model, t_slider, fn_in, fp_in]
        t_outputs = [cm_plot, cost_plot, summary_md]
        for comp in t_inputs:
            comp.change(threshold_view, t_inputs, t_outputs)

    def init_view():
        cm, cost, summary = threshold_view("sklearn", VAL_THRESHOLDS["sklearn"], DEFAULT_FN_COST, DEFAULT_FP_COST)
        return (plot_pred_vs_actual("sklearn"), plot_calibration("sklearn"),
                plot_feature(feature_cols[0]), cm, cost, summary)

    demo.load(init_view, None, [pva_plot, calib_plot, feat_plot, cm_plot, cost_plot, summary_md])


if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
